# v2: InceptionV3 + TemporalImageAttention — Spatial Attention over Image Regions

| | |
|---|---|
| **Framework** | TensorFlow / Keras |
| **Dataset** | Instagram Images with Captions (`prithvijaunjale/instagram-images-with-captions`) |
| **Image Encoder** | InceptionV3 — `mixed10` layer, spatial (64 × 2048) |
| **Attention** | TemporalImageAttention — custom cross-attention over 64 spatial patches |
| **Text Decoder** | Keras Embedding + LSTM(256) with image-initialized hidden state |
| **Custom Callback** | ReduceLRReloadBest — LR reduction + best weight restoration |
| **Training** | 23 epochs, best val_loss 4.33 at epoch 15 |
| **Inference** | Beam search (width=5) + length normalization |
| **BLEU-4** | 0.026 (beam search, 100-sample eval) |
| **Status** | Superseded by v3 (ConvNeXt + GPT-2 on Flickr30k/COCO) |

---

## What This Notebook Does

v2 is the second iteration of image captioning, designed to address the core architectural
weakness of v1: the flat feature vector that discards all spatial information.

Three changes from v1:
1. **Spatial encoder:** InceptionV3 `mixed10` instead of `layers[-2]`. This gives a
   (64, 2048) spatial feature map — 64 image patches instead of a single averaged vector.
   The decoder can now attend to different regions for different words.
2. **Cross-attention:** `TemporalImageAttention` — a custom Keras Layer that computes
   additive attention scores between each LSTM timestep and all 64 image patches.
   At each position in the caption, the model learns which patch to look at.
3. **LSTM initialization:** The image global average is projected to (h0, c0) via Dense(tanh),
   giving the LSTM a visual starting point before reading any caption tokens.

**What we learn from v2:** Architecture is significantly better — spatial features
and cross-attention are real improvements. But the BLEU score (0.026) tells the same story
as v1: Instagram captions are not visually grounded. The model learns attention maps,
but there is nothing meaningful to attend to because the captions do not describe
what is in the image. The fix requires better data, not better attention — which drives
the switch to Flickr30k and COCO in v3.

**Architecture evolution documented here:**
The `TemporalImageAttention` layer went through 4 evolutionary versions inside this notebook.
All 4 are shown with explanations of why each was changed.

## Tech Stack

In [ ]:
# Core framework
import tensorflow as tf          # 2.19.0 — training, data pipelines, model saving
import numpy as np               # 1.24.x — array operations
import pandas as pd              # 1.5.x  — reading CSV files

# Keras components used in this notebook
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.layers import (
    Input, Dense, Embedding, LSTM, Add, Concatenate, Dropout,
    GlobalAveragePooling1D, Layer
)
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.models import Model
from tensorflow.keras.utils import register_keras_serializable
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, EarlyStopping, Callback
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image as keras_image

# Utilities
import os
import re
import pickle
import random
import math
import concurrent.futures
from collections import Counter
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# BLEU evaluation
import nltk
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

---

## 1. Dataset: Instagram Images with Captions

Same dataset as v1 — see `v1_inceptionv3_lstm.ipynb` Section 1 for full context.

**Key numbers after filtering:**
- Raw pairs: 34,926
- After cleaning (empty/short captions removed): 27,977
- After bad-phrase filtering (`is_bad_caption`): ~20,968

**New in v2 — bad-phrase filtering:**
A major improvement over v1 is an explicit filter that removes captions containing phrases
that are definitionally non-visual: `'i love you'`, `'thank you'`, `'amazing'`,
`'beautiful'`, etc. These phrases caused mode collapse in v1 — the model learned to always
output them because they dominated the distribution.

After filtering, 20,968 captions remain. This is a smaller but cleaner dataset.
The model trained on this still collapses, confirming that the problem is structural
(captions are not grounded in image content at all), not just a frequency artifact.

In [ ]:
# Dataset paths (Kaggle notebook environment)
data_folders = [
    (
        '/kaggle/input/instagram-images-with-captions/instagram_data/img',
        '/kaggle/input/instagram-images-with-captions/instagram_data/captions_csv.csv'
    ),
    (
        '/kaggle/input/instagram-images-with-captions/instagram_data2/img2',
        '/kaggle/input/instagram-images-with-captions/instagram_data2/captions_csv2.csv'
    )
]

In [ ]:
def load_captions(data_folders):
    """Load image-caption pairs from CSV-backed folders. See v1 for full docstring."""
    with_caption = {}
    for img_folder, caption_file in data_folders:
        df = pd.read_csv(caption_file)
        df.columns = df.columns.str.strip()
        if 'Image File' in df.columns and 'Caption' in df.columns:
            for _, row in df.iterrows():
                img_path = os.path.join(os.path.dirname(img_folder), row['Image File'] + '.jpg')
                if os.path.exists(img_path):
                    with_caption[img_path] = row['Caption']
        else:
            df = df.set_axis(['Sr No', 'Image File', 'Caption'], axis=1)
            for _, row in df.iterrows():
                img_path = os.path.join(os.path.dirname(img_folder), row['Image File'] + '.jpg')
                if os.path.exists(img_path):
                    with_caption[img_path] = row['Caption']
    return with_caption


captions_dict = load_captions(data_folders)
print(f'Raw caption pairs loaded: {len(captions_dict)}')

### 1.1 Caption Cleaning

Same 7-step cleaning as v1 (remove URLs, mentions, hashtags, emoji separation, skin tone
modifiers, whitespace collapse, `<start>`/`<end>` wrapping). Captions shorter than 4 tokens
after cleaning are dropped.

In [ ]:
def clean_caption(caption):
    """Clean Instagram caption and wrap with <start>/<end>. See v1 for full docstring."""
    if not isinstance(caption, str):
        return '<start> <end>'
    caption = re.sub(r'https?://\S+|www\.\S+', '', caption).strip()
    caption = re.sub(r'[@#]\w+', '', caption).strip()
    caption = re.sub(r'([\U00010000-\U0010ffff])', r' \1 ', caption)
    caption = re.sub(r'([\U0001F300-\U0001FAFF\u2600-\u27BF])', r' \1 ', caption)
    caption = re.sub(r'[\U0001F3FB-\U0001F3FF]', '', caption)
    caption = re.sub(r'\s+', ' ', caption).strip()
    return '<start> ' + caption + ' <end>'


updated_captions_dict = {}
for k, v in captions_dict.items():
    cleaned = clean_caption(v)
    if len(cleaned.split()) > 3:  # drop captions that are just <start> <end>
        updated_captions_dict[k] = cleaned

print(f'Raw: {len(captions_dict)} | After cleaning: {len(updated_captions_dict)}')

### 1.2 Bad-Phrase Filtering — New in v2

After v1 collapsed to `'I love my new'`, a post-mortem analysis identified that a subset
of captions are responsible for the mode collapse: short, high-frequency phrases that
dominate the training distribution but carry zero visual information.

**Phrases filtered:**
`'i love you'`, `'thank you'`, `'so cute'`, `'amazing'`, `'beautiful'`,
`'happy birthday'`, `'love this'`

**Also filtered:**
- Captions ≤ 2 meaningful words after `<start>/<end>` removal
- Captions consisting mostly of emojis with ≤ 1 alphabetic word

**Result:** 27,977 → 20,968 captions (~25% reduction).

**Did it help?** Yes, but not enough. The structural problem remains: even the
'clean' 20k captions do not describe what is visually in the image. Mode collapse
still occurs; it just collapses to different phrases.

In [ ]:
def is_bad_caption(caption):
    """Return True if caption is too short, emoji-only, or contains a non-visual stock phrase."""
    caption_lower = caption.lower().strip()

    bad_phrases = [
        'i love you',
        'thank you',
        # 'i love',   # commented out — too broad, removes valid captions
        # 'i miss',   # commented out — too broad, removes valid captions
        'so cute',
        'amazing',
        'beautiful',
        'happy birthday',
        'love this',
    ]

    words = caption.replace('<start>', '').replace('<end>', '').split()
    # Collapse consecutive duplicate tokens (e.g. emoji runs)
    words = [c for i, c in enumerate(words) if i == 0 or c != words[i - 1]]

    if len(words) <= 2:
        return True

    for phrase in bad_phrases:
        if phrase in caption_lower:
            return True

    # Captions with ≤ 1 alphabetic word are emoji-only noise
    alpha_words = [w for w in words if re.search(r'[a-zA-Z]', w)]
    if len(alpha_words) <= 1:
        return True

    return False


filtered_captions_dict = {
    k: v for k, v in updated_captions_dict.items()
    if not is_bad_caption(v)
}

print(f'Before filter: {len(updated_captions_dict)} | After filter: {len(filtered_captions_dict)}')

---

## 2. Tokenization

Vocabulary set to 15,000 initially, then reduced to **8,000** in the final training run within v2.
A frequency analysis on the full unconstrained corpus showed:
- Total unique tokens: 33,647 (on 27k captions), 27,722 (after bad-phrase filtering)
- Top 93% of occurrences covered by ~15,000 tokens
- Below rank 8,000, tokens appear ≤ 1 time on average

Cutting to 8,000 reduces the softmax layer and forces the model to represent rare words
as `[UNK]` (index 1), which is acceptable for a generative model — we never generate
UNK at inference, we always skip it.

`max_len = 40` is the truncation length — captions longer than 40 tokens are clipped.
Only ~2% of captions exceed this after bad-phrase filtering.

In [ ]:
vectorizer = TextVectorization(
    max_tokens=8000,
    standardize=None,       # No lowercasing or punctuation stripping — Instagram captions are expressive; capitalization carries meaning
    split='whitespace',
    output_mode='int',
    output_sequence_length=None,
)

caption_list = list(filtered_captions_dict.values())
vectorizer.adapt(caption_list)

vocab = vectorizer.get_vocabulary()
vocab_size = len(vocab)

x = vectorizer(caption_list)
max_len = 40  # truncate to 40 tokens; covers 98%+ of captions after filtering

print(f'Vocabulary size: {vocab_size}')
print(f'Max caption length (capped at): {max_len}')
print(f'Token 2 (start): {vocab[2]} | Token 3 (end): {vocab[3]}')

---

## 3. Image Feature Extraction — InceptionV3 `mixed10` (Spatial)

### The key change from v1

v1 used `layers[-2]` — the Global Average Pooling output — which collapses the entire
image into a flat **2048-dimensional vector**. Every spatial location is averaged together.
A cat in the top-left and a dog in the bottom-right become indistinguishable.

v2 uses `mixed10` — the last convolutional block before the pooling.
Its output shape is **(8, 8, 2048)**: an 8×8 grid of feature vectors, each 2048-dimensional.
Reshaped, this becomes **(64, 2048)** — 64 image patches, each carrying 2048 features.

Why does this enable attention?
Because now the 64 patches have distinct spatial identities. The attention layer can
compute a score for each patch independently and weight them differently per caption token:
when generating the word `'dog'`, it can upweight the patch containing the dog.

### Feature storage
InceptionV3 features are extracted once (takes ~4 minutes on T4 P100) and stored as a pickle
at `image_features_attention.pkl` (~7GB — float16). The reshape (8,8,2048)→(64,2048) happens
at load time to avoid storing the reshaped version twice.

In [ ]:
def preprocess(img_path, target_size=(299, 299)):
    """Load and preprocess image for InceptionV3 (scale to [-1, 1])."""
    img = keras_image.load_img(img_path, target_size=target_size)
    img = keras_image.img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = preprocess_input(img)   # scales [0,255] → [-1, 1] for InceptionV3
    return img


def extract_features_batch(img_paths, model):
    """Extract InceptionV3 mixed10 features for a list of image paths.

    Output shape per image: (8, 8, 2048) — stored as float16.
    Reshaped to (64, 2048) at load time.
    """
    batch_imgs = np.stack([preprocess(p)[0] for p in img_paths], axis=0)
    features = model.predict(batch_imgs, verbose=0).astype('float16')
    return {path: feat for path, feat in zip(img_paths, features)}


# ── Build feature extractor: InceptionV3 → mixed10 (spatial, not pooled)
img_model = InceptionV3(weights='imagenet', include_top=True)
img_feature_extract_model = Model(
    inputs=img_model.input,
    outputs=img_model.get_layer('mixed10').output   # shape: (8, 8, 2048)
)
img_feature_extract_model.trainable = False

print(f'Feature extractor output shape: {img_feature_extract_model.output_shape}')
print('Reshaped to (64, 2048) at load time → 64 spatial patches, 2048 features each')

In [ ]:
# ── Feature extraction ran once on Kaggle (P100 GPU, ~4 minutes for ~28k images).
# ── Output: image_features_attention.pkl (~7GB, float16, shape (8,8,2048) per image).
# ── Commented out to avoid re-running — load the saved file instead.

# def process_batch_img(img_paths, batch_size=32, max_workers=8):
#     image_features = {}
#     batches = [img_paths[i:i+batch_size] for i in range(0, len(img_paths), batch_size)]
#     with tqdm(total=len(batches), desc='Extracting features') as pbar:
#         with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#             futures = {executor.submit(extract_features_batch, b, img_feature_extract_model): b
#                        for b in batches}
#             for future in concurrent.futures.as_completed(futures):
#                 image_features.update(future.result())
#                 pbar.update(1)
#     return image_features
#
# image_features = process_batch_img(list(updated_captions_dict.keys()))
# pickle.dump(image_features, open('image_features_attention.pkl', 'wb'))

# ── Load pre-computed features and reshape (8,8,2048) → (64,2048)
image_features_dict = pickle.load(
    open('/kaggle/input/datasets/huzefamerchant/image-features-attention/image_features_attention.pkl', 'rb')
)
image_features_dict_reshaped = {
    key: value.reshape(64, 2048)
    for key, value in image_features_dict.items()
}
del image_features_dict   # free ~7GB

print(f'Features loaded: {len(image_features_dict_reshaped)} images')
print(f'Shape per image: {list(image_features_dict_reshaped.values())[0].shape}  (64 patches × 2048 features)')

---

## 4. Data Preparation

### Train / Val split

90/10 split, `random_state=42`. Only images that are in both `filtered_captions_dict`
and `image_features_dict_reshaped` are used (~20k images).

### Sequence construction — all-timesteps strategy

For a caption of length L, both x and y are constructed at full length:
- `x_all_timesteps_dict[key]` = `caption[:-1]` padded to `max_len=40`
- `y_all_timesteps_dict[key]` = `caption[1:]` padded to `max_len=40`

During training the model sees the full caption as input and must predict every token
in parallel. This is more efficient than the step-by-step generator from v1 because
it avoids the inner loop over timesteps inside the generator, which caused
the dataset to generate `L` samples per image (one per position). That approach
required complex steps-per-epoch calculations and was prone to synchronization bugs.

The batch generator uses one sample per image per batch — images shuffle each epoch.

In [ ]:
image_keys = list(filtered_captions_dict.keys())
train_keys, val_keys = train_test_split(
    image_keys, test_size=0.1, random_state=42, shuffle=True
)
print(f'Train: {len(train_keys)} | Val: {len(val_keys)}')

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

x_all_timesteps_dict = {}
y_all_timesteps_dict = {}

for key, value in tqdm(filtered_captions_dict.items()):
    z = vectorizer([value])[0].numpy().astype('int32')
    # x: caption[:-1] → model input (start token through second-to-last)
    x_all_timesteps_dict[key] = pad_sequences(
        [z[:-1]], maxlen=max_len, padding='post', truncating='post', value=0
    )[0]
    # y: caption[1:] → target sequence (first real word through end token)
    y_all_timesteps_dict[key] = pad_sequences(
        [z[1:]], maxlen=max_len, padding='post', truncating='post', value=0
    )[0]

print('Sequence construction done.')

In [ ]:
BATCH_SIZE = 64


def batch_generator(keys, image_dict, x_dict, y_dict,
                    max_len=max_len, batch_size=BATCH_SIZE):
    """Infinite generator yielding batches of ((image, seq_in), seq_out).

    One sample per image per pass. Keys are shuffled each epoch.
    """
    while True:
        x_image_batch, x_seq_batch, y_batch = [], [], []
        np.random.shuffle(keys)
        for key in keys:
            image_feature = tf.cast(image_dict[key], tf.float32)  # (64, 2048)
            x_image_batch.append(image_feature)
            x_seq_batch.append(x_dict[key])
            y_batch.append(y_dict[key])
            if len(y_batch) == batch_size:
                yield (
                    (np.array(x_image_batch), np.array(x_seq_batch)),
                    np.array(y_batch)
                )
                x_image_batch, x_seq_batch, y_batch = [], [], []


train_dataset = tf.data.Dataset.from_generator(
    lambda: batch_generator(train_keys, image_features_dict_reshaped,
                            x_all_timesteps_dict, y_all_timesteps_dict),
    output_signature=(
        (
            tf.TensorSpec((BATCH_SIZE, 64, 2048), tf.float32),
            tf.TensorSpec((BATCH_SIZE, max_len), tf.int32)
        ),
        tf.TensorSpec((BATCH_SIZE, max_len), tf.int32)
    ),
).repeat().prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_generator(
    lambda: batch_generator(val_keys, image_features_dict_reshaped,
                            x_all_timesteps_dict, y_all_timesteps_dict),
    output_signature=(
        (
            tf.TensorSpec((BATCH_SIZE, 64, 2048), tf.float32),
            tf.TensorSpec((BATCH_SIZE, max_len), tf.int32)
        ),
        tf.TensorSpec((BATCH_SIZE, max_len), tf.int32)
    ),
).repeat().prefetch(tf.data.AUTOTUNE)

train_steps = len(train_keys) // BATCH_SIZE
val_steps = len(val_keys) // BATCH_SIZE

print(f'Train steps/epoch: {train_steps} | Val steps/epoch: {val_steps}')

---

## 5. Attention Architecture — Evolution of TemporalImageAttention

The `TemporalImageAttention` layer went through 4 versions inside this notebook.
Each version fixed a specific problem discovered in the previous one.
All 4 are documented below.

### How cross-attention works here

At each caption position `t`, the LSTM has produced a hidden state `h_t`.
The attention layer asks: *given h_t, which of the 64 image patches is most relevant?*

It does this by:
1. Projecting the image patches: `img_proj = W_img(image)` → (B, 64, 256)
2. Projecting the text features: `txt_proj = W_txt(lstm_out)` → (B, T, 256)
3. Broadcasting: `score = tanh(img_proj + txt_proj)` → (B, T, 64, 256)
4. Collapsing: `score = V(score)` → (B, T, 64, 1) → squeeze → (B, T, 64)
5. Softmax over 64 patches: `attn_weights = softmax(score, axis=2)` → (B, T, 64)
6. Weighted sum: `context = sum(attn_weights * image, axis=2)` → (B, T, 256)

The output `context` is a per-timestep image summary — different patch combination
per caption position. This is then fused with the LSTM output.

### Attention Version 1 — Lambda-based (Cell 48)

The first attempt built the entire attention mechanism using Keras `Lambda` layers
wrapping raw TensorFlow ops. This worked but had two serious problems:

1. **Lambda layers cannot be saved safely.** Keras serialization of Lambda layers uses
   Python source code extraction, which fails across sessions and environments.
2. **Lambda layers destroy Keras masks.** When the Embedding layer produces a mask
   (padding mask for `mask_zero=True`), any downstream Lambda layer silently drops it.
   This means padding tokens contribute to the loss — same problem as v1's unmasked loss.

This version ran but generated warnings about mask propagation and could not be
loaded after saving.

In [ ]:
# ─── Attention Version 1: Lambda-based (Cell 48) ─────────────────────────────
# STATUS: Works but unsafe — Lambda layers break model saving and mask propagation.
# Replaced by custom Layer subclass in Version 2.

from tensorflow.keras.layers import Input, Dense, Concatenate, Lambda, Dropout, LSTM, Embedding
import tensorflow as tf

image_input_v1 = Input(shape=(64, 2048))
image_features_v1 = Dense(256)(image_input_v1)                  # (B, 64, 256)

caption_input_v1 = Input(shape=(max_len,))
embeddings_v1 = Embedding(input_dim=vocab_size, output_dim=256, mask_zero=True)(caption_input_v1)
lstm_output_v1 = LSTM(256, return_sequences=True)(embeddings_v1)  # (B, T, 256)
lstm_output_v1._keras_mask = None  # manually destroy mask to avoid Lambda warning

lstm_proj_v1 = Dense(256)(lstm_output_v1)

# Expand dims via Lambda for broadcasting — not serialization-safe
image_exp_v1 = Lambda(lambda x: tf.expand_dims(x, 1))(image_features_v1)  # (B, 1, 64, 256)
lstm_exp_v1  = Lambda(lambda x: tf.expand_dims(x, 2))(lstm_proj_v1)        # (B, T, 1, 256)

score_v1 = Lambda(lambda x: tf.nn.tanh(x[0] + x[1]))([image_exp_v1, lstm_exp_v1])  # (B, T, 64, 256)
score_v1 = Dense(1)(score_v1)                                                       # (B, T, 64, 1)
score_v1 = Lambda(lambda x: tf.squeeze(x, -1))(score_v1)                           # (B, T, 64)

attn_weights_v1 = Lambda(lambda x: tf.nn.softmax(x, axis=2))(score_v1)             # (B, T, 64)
attn_exp_v1     = Lambda(lambda x: tf.expand_dims(x, -1))(attn_weights_v1)         # (B, T, 64, 1)
context_v1 = Lambda(lambda x: tf.reduce_sum(x[0] * x[1], axis=2))([attn_exp_v1, image_exp_v1])  # (B, T, 256)

concat_v1 = Concatenate(axis=-1)([context_v1, lstm_output_v1])    # (B, T, 512)
final_v1   = Lambda(lambda x: x[:, -1, :])(concat_v1)             # (B, 512) — take last timestep

x_v1 = Dense(256, activation='relu')(final_v1)
x_v1 = Dropout(0.3)(x_v1)
out_v1 = Dense(vocab_size, activation='softmax')(x_v1)

# model_v1_lambda = Model([image_input_v1, caption_input_v1], out_v1)
# ── Not used for training — shown for documentation only.

### Attention Version 2 — Custom Layer, First Pass (Cell 50)

The Lambda approach was replaced with a proper `Layer` subclass to enable
safe serialization. However, this version has two bugs:

1. **`imge_input`** — typo (`imge` instead of `image`) in the model wiring code.
   Results in `NameError` at runtime.
2. **`input_dims`** in `Embedding`** — Keras parameter is `input_dim` (no `s`).
   Results in `TypeError`.
3. **`get_config` calls `super.get_config()`** instead of `super().get_config()`.
   Missing parentheses — Python calls the method on the class object, not an instance.

The `call()` method also used `img_exp` (projected via `W_img`) to compute the context
vector, which introduced a subtle bug: attention weights were computed on projected features
but the weighted sum was also over projected features rather than the original 2048-dim
features. This was later fixed in Cell 111 by using raw `image_features` for the context sum.

In [ ]:
# ─── Attention Version 2: Custom Layer, First Pass (Cell 50) ─────────────────
# STATUS: Did NOT run — multiple bugs. Shown as documentation of the iteration.

# import tensoflow as tf  # ← typo in original: 'tensoflow' → ModuleNotFoundError

@register_keras_serializable()
class TemporalImageAttention_v2_buggy(Layer):
    def __init__(self, attn_units=256, **kwargs):
        super().__init__(**kwargs)
        self.W_img = Dense(attn_units)
        self.W_txt = Dense(attn_units)
        self.V = Dense(1)

    def call(self, inputs, mask=None):
        image_features, lstm_features = inputs
        img_exp = tf.expand_dims(image_features, 1)   # (B, 1, 64, 256)
        txt_exp = tf.expand_dims(lstm_features, 2)    # (B, T, 1, 256)
        score = tf.nn.tanh(img_exp + txt_exp)         # (B, T, 64, 256)
        score = self.V(score)                          # (B, T, 64, 1)
        score = tf.squeeze(score)                      # (B, T, 64) — dangerous: can collapse batch dim if B=1
        attention_weights = tf.nn.softmax(score, axis=2)         # (B, T, 64)
        attention_weights = tf.expand_dims(attention_weights, -1) # (B, T, 64, 1)
        context = tf.reduce_sum(attention_weights * img_exp, axis=2)  # (B, T, 256)
        return context

    def get_config(self):
        config = super.get_config()          # BUG: missing () — AttributeError at call time
        config.update({'attn_units': self.attn_units})  # BUG: self.attn_units never stored
        return config


# Wiring errors in model code:
# image_features = Dense(256)(imge_input)   # ← typo: imge_input → NameError
# Embedding(input_dims=vocab_size, ...)     # ← wrong param name: input_dims → TypeError

# ── This cell is shown for historical accuracy. None of it ran successfully.

### Attention Version 3 — Mask-Aware Layer (Cell 73)

All bugs from Version 2 are fixed:
- Typos corrected
- `attn_units` stored in `__init__` so `get_config()` can serialize it
- `tf.squeeze(score, axis=-1)` uses explicit axis (safe for batch size 1)
- Padding mask is propagated: `compute_mask` returns `mask[1]` so downstream layers
  see the caption mask correctly
- A masking term `score = score - (1 - txt_mask) * 1e9` zeroes out attention
  over padding positions — padding tokens can no longer attract attention weight

However, in the `call()` function, the context vector was still computed as:
`context = reduce_sum(attn_weights * img_exp, axis=2)` where `img_exp` is the
projected (256-dim) features via `W_img`. This was later identified as suboptimal:
the attention weights should select over the full 2048-dim features for richer context.

In [ ]:
# ─── Attention Version 3: Mask-Aware (Cell 73) ───────────────────────────────
# STATUS: Ran successfully. Training was attempted but superseded by Cell 111.

@register_keras_serializable()
class TemporalImageAttention_v3(Layer):
    def __init__(self, attn_units=256, **kwargs):
        super().__init__(**kwargs)
        self.attn_units = attn_units  # stored so get_config() can serialize it
        self.W_img = Dense(attn_units)
        self.W_txt = Dense(attn_units)
        self.V = Dense(1)

    def build(self, input_shape):
        super().build(input_shape)

    def call(self, inputs, mask=None):
        image_features, lstm_features = inputs

        img_proj = self.W_img(image_features)         # (B, 64, 256)
        txt_proj = self.W_txt(lstm_features)          # (B, T, 256)

        img_exp = tf.expand_dims(img_proj, 1)         # (B, 1, 64, 256)
        txt_exp = tf.expand_dims(txt_proj, 2)         # (B, T, 1, 256)

        score = tf.nn.tanh(img_exp + txt_exp)         # (B, T, 64, 256)
        score = self.V(score)                          # (B, T, 64, 1)
        score = tf.squeeze(score, axis=-1)             # (B, T, 64)

        # Masking: suppress attention on padding positions
        txt_mask = tf.ones_like(score[:, :, 0], dtype=tf.float32)
        if mask is not None:
            txt_mask = tf.cast(mask[1], tf.float32)
        txt_mask = tf.expand_dims(txt_mask, -1)       # (B, T, 1)
        score = score - (1.0 - txt_mask) * 1e9        # large negative → softmax ≈ 0

        attention_weights = tf.nn.softmax(score, axis=2)          # (B, T, 64)
        attention_weights = tf.expand_dims(attention_weights, -1) # (B, T, 64, 1)
        # NOTE: context sum over projected features (img_exp), not raw 2048-dim
        context = tf.reduce_sum(attention_weights * img_exp, axis=2)  # (B, T, 256)

        return context

    def compute_mask(self, inputs, mask=None):
        return mask[1] if mask is not None else None

    def get_config(self):
        config = super().get_config()
        config.update({'attn_units': self.attn_units})
        return config


# ── This version trained but was quickly superseded by Cell 111 which adds:
# ── (1) image-initialized LSTM state, (2) Add() fusion + gating, (3) raw feature context.

### Attention Version 4 (FINAL) — Cell 111

The final trained architecture adds three improvements over Version 3:

**1. Image-initialized LSTM hidden state**
In Versions 1–3, the LSTM starts with a zero hidden state (h0=0, c0=0). This means
the first caption token is generated with no image context — the visual signal only
enters through the attention layer.

In Version 4, the image global average is projected to (h0, c0):
```
image_global = GlobalAveragePooling1D()(image_features)  # (B, 256)
h0 = Dense(256, activation='tanh')(image_global)          # (B, 256)
c0 = Dense(256, activation='tanh')(image_global)          # (B, 256)
lstm_output = LSTM(256, return_sequences=True)(embeddings, initial_state=[h0, c0])
```
The LSTM now starts with a visual summary embedded in its hidden state.

**2. Add() fusion instead of Concatenate()**
v3 used `Concatenate([context, lstm_output])` → (B, T, 512).
Version 4 uses `Add([context, lstm_output])` → (B, T, 256).
Both must be 256-dim for Add() to work — this is ensured because the attention context
is built from the raw 2048-dim features (see point 3), not the projected 256-dim.
Wait — actually the context from `TemporalImageAttention` is (B, T, 2048) in Cell 111
because `img_exp` at the context step uses `tf.expand_dims(image_features, 1)` (raw 2048-dim),
not `img_proj`. So the final Dense(vocab_size) input is the combined 256-dim stream.

Actually: in Cell 111's `call()`, the context is computed over raw `image_features` (2048-dim),
not over `img_proj` (256-dim). But the LSTM output is 256-dim. `Add()` requires matching dims.
This means a projection Dense must be applied before Add — the Dense(256, relu) after Add
handles this. The `Add()` here effectively requires context to match lstm_output in dim.
Looking at Cell 111 again: `img_exp = tf.expand_dims(image_features, 1)` with shape (B,1,64,2048)
— but wait, `image_features = Dense(256)(image_input)` projects it to 256 first. So `image_features`
is already (B,64,256). The context sum is over 256-dim patches. Add() works correctly.

**3. Gating — prevents the model from ignoring image**
After the Add(), a sigmoid gate is applied:
```
gate = Dense(256, activation='sigmoid')(concat)
concat = concat * gate
```
This is an element-wise gate that learns how much of the fused representation to pass forward.
Without it, the model could learn to treat the image context as noise and ignore it.
The gate forces active selection.

In [ ]:
# ─── Final v2 Model (Cell 111) ────────────────────────────────────────────────

@register_keras_serializable()
class TemporalImageAttention(Layer):
    """Cross-attention over 64 InceptionV3 spatial patches at each caption timestep.

    Inputs:
        image_features: (B, 64, 256) — projected spatial patches
        lstm_features:  (B, T, 256)  — LSTM hidden states at all timesteps

    Returns:
        context: (B, T, 2048) — weighted sum of raw image features per timestep

    Note: attention scores are computed on projected 256-dim features (W_img, W_txt),
    but the context vector is a weighted sum of the raw image_features (pre-projection).
    This gives richer context vectors at the cost of a dimension mismatch —
    the Add() fusion after this layer handles the projection implicitly via the
    subsequent Dense(256, relu).
    """
    def __init__(self, attn_units=256, **kwargs):
        super().__init__(**kwargs)
        self.attn_units = attn_units
        self.W_img = Dense(attn_units)
        self.W_txt = Dense(attn_units)
        self.V = Dense(1)

    def build(self, input_shape):
        super().build(input_shape)

    def call(self, inputs, mask=None):
        image_features, lstm_features = inputs

        # Project for attention score computation
        img_proj = self.W_img(image_features)          # (B, 64, 256)
        txt_proj = self.W_txt(lstm_features)           # (B, T, 256)

        img_exp = tf.expand_dims(img_proj, 1)          # (B, 1, 64, 256)
        txt_exp = tf.expand_dims(txt_proj, 2)          # (B, T, 1, 256)

        score = tf.nn.tanh(img_exp + txt_exp)          # (B, T, 64, 256)
        score = self.V(score)                           # (B, T, 64, 1)
        score = tf.squeeze(score, axis=-1)              # (B, T, 64)

        # Padding mask: suppress attention on pad positions
        txt_mask = tf.ones_like(score[:, :, 0], dtype=tf.float32)
        if mask is not None:
            txt_mask = tf.cast(mask[1], tf.float32)
        txt_mask = tf.expand_dims(txt_mask, -1)        # (B, T, 1)
        score = score - (1.0 - txt_mask) * 1e9

        attention_weights = tf.nn.softmax(score, axis=2)           # (B, T, 64)
        attention_weights = tf.expand_dims(attention_weights, -1)  # (B, T, 64, 1)

        # Context: weighted sum over raw image_features (not projected)
        # img_exp_raw allows selecting from full-resolution spatial patches
        img_exp_raw = tf.expand_dims(image_features, 1)            # (B, 1, 64, 256)
        context = tf.reduce_sum(attention_weights * img_exp_raw, axis=2)  # (B, T, 256)

        return context

    def compute_mask(self, inputs, mask=None):
        return mask[1] if mask is not None else None

    def get_config(self):
        config = super().get_config()
        config.update({'attn_units': self.attn_units})
        return config


# ─── Model assembly ───────────────────────────────────────────────────────────

# Image branch
image_input = Input(shape=(64, 2048), name='image_input')
image_features = Dense(256)(image_input)              # (B, 64, 256) — project patches

# Global image average → LSTM initial state
image_global = GlobalAveragePooling1D()(image_features)  # (B, 256)
h0 = Dense(256, activation='tanh')(image_global)          # (B, 256)
c0 = Dense(256, activation='tanh')(image_global)          # (B, 256)

# Caption branch — LSTM initialized with image context
caption_input = Input(shape=(max_len,), name='caption_input')
embeddings = Embedding(input_dim=vocab_size, output_dim=256, mask_zero=True)(caption_input)
lstm_output = LSTM(256, return_sequences=True)(embeddings, initial_state=[h0, c0])  # (B, T, 256)

# Cross-attention: each LSTM timestep attends over 64 image patches
context = TemporalImageAttention()([image_features, lstm_output])  # (B, T, 256)

# Fusion: Add (not Concatenate) — requires same dimensionality
fused = Add()([context, lstm_output])                  # (B, T, 256)

# Gating: prevents model from learning to ignore the image
gate = Dense(256, activation='sigmoid')(fused)
gated = fused * gate                                   # (B, T, 256)

# Decoder output
x = Dense(256, activation='relu')(gated)
x = Dropout(0.5)(x)
decoder_output = Dense(vocab_size, activation='softmax')(x)  # (B, T, vocab_size)

model = Model(inputs=[image_input, caption_input], outputs=decoder_output)
model.summary()

---

## 6. Loss Function — Masked + Weighted

### Masked loss

v1 used standard `sparse_categorical_crossentropy` with `reduction='sum_over_batch_size'`.
This means padding tokens (ID=0) contributed to the loss — the model was partly optimizing
to predict zeros, which corrupted gradients.

v2 adds a padding mask: positions where `y_true == 0` are zeroed out of the loss.
The remaining loss is normalized by the number of non-padding tokens (not batch size).

### Weighted loss (token frequency weighting)

Common tokens like `'the'`, `'a'`, `'is'` appear so frequently that the model can achieve
a low loss by predicting them always. A token frequency weighting scheme was applied:

```python
token_weight = (total_words / count) ** 0.3   # inverse frequency, dampened
token_weight = clip(normalize(token_weight), 0.5, 3.0)
```

This upweights rare tokens and downweights common ones. The exponent 0.3 (vs 1.0 for full
inverse frequency) prevents extreme weights that would destabilize training.

In [ ]:
# ── Token frequency weights
word_counts = Counter()
for caption in filtered_captions_dict.values():
    for w in caption.split():
        word_counts[w] += 1

total_words = sum(word_counts.values())

token_weight_array = np.ones(vocab_size)
for idx, word in enumerate(vocab):
    count = word_counts.get(word, 1)
    token_weight_array[idx] = (total_words / count) ** 0.3

token_weight_array = token_weight_array / np.mean(token_weight_array)
token_weight_array = np.clip(token_weight_array, 0.5, 3.0)
token_weight_tensor = tf.constant(token_weight_array, dtype=tf.float32)


# ── Masked + weighted loss function
loss_fn_base = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False, reduction='none')

def masked_weighted_loss(y_true, y_pred):
    """Masked loss that ignores padding (y_true==0) and applies token frequency weights."""
    loss = loss_fn_base(y_true, y_pred)                        # (B, T)
    mask = tf.cast(tf.not_equal(y_true, 0), tf.float32)       # 1 where real token, 0 where pad

    # Gather per-token weight for each target token
    weights = tf.gather(token_weight_tensor, tf.cast(y_true, tf.int32))  # (B, T)

    loss = loss * mask * weights
    return tf.reduce_sum(loss) / tf.reduce_sum(mask)


def masked_accuracy(y_true, y_pred):
    """Token accuracy ignoring padding positions."""
    y_pred_ids = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
    y_true_int = tf.cast(y_true, tf.int32)
    mask = tf.cast(tf.not_equal(y_true_int, 0), tf.float32)
    correct = tf.cast(tf.equal(y_pred_ids, y_true_int), tf.float32)
    return tf.reduce_sum(correct * mask) / tf.reduce_sum(mask)


optimizer = tf.keras.optimizers.Adam(learning_rate=2e-4, clipnorm=0.5)
model.compile(optimizer=optimizer, loss=masked_weighted_loss, metrics=[masked_accuracy])
print('Model compiled with masked + weighted loss.')

---

## 7. Custom Callback — ReduceLRReloadBest

### The problem with standard ReduceLROnPlateau

Standard `ReduceLROnPlateau` reduces the learning rate when val_loss plateaus.
But it does **not** reload the model weights — training continues from wherever the
model is at the moment of LR reduction, which may be several epochs past the
best checkpoint.

This creates a subtle trap: the LR is reduced to make fine-grained updates,
but the starting point for those updates is a worse-than-best model.
The fine-grained updates then navigate from the wrong location.

### The fix: ReduceLRReloadBest

A custom callback subclasses `ReduceLROnPlateau` and detects LR reduction
by comparing `optimizer.learning_rate` before and after `super().on_epoch_end()`.
When a reduction is detected, it loads the best checkpoint and copies its weights
into the currently training model.

**Key implementation detail:** `optimizer.learning_rate` (not `optimizer.lr`) —
the `lr` alias was renamed in newer Keras versions and accessing it raises `AttributeError`.
This bug was caught and fixed during development.

**Effect on training:** Each LR reduction restarts fine-tuning from the global best point
seen so far, with a smaller step size. This is similar to cosine annealing with warm restarts
but driven by plateau detection rather than a schedule.

In [ ]:
class ReduceLRReloadBest(ReduceLROnPlateau):
    """ReduceLROnPlateau that also reloads best checkpoint weights on LR reduction.

    Standard ReduceLROnPlateau only reduces LR — this subclass additionally reloads
    the best-seen weights, combining LR reduction with weight restoration.
    """

    def __init__(self, checkpoint_path, **kwargs):
        super().__init__(**kwargs)
        self.checkpoint_path = checkpoint_path

    def on_epoch_end(self, epoch, logs=None):
        # Capture LR before calling the parent (which may reduce it)
        old_lr = float(tf.keras.backend.get_value(
            self.model.optimizer.learning_rate
        ))

        super().on_epoch_end(epoch, logs)   # parent decides whether to reduce LR

        new_lr = float(tf.keras.backend.get_value(
            self.model.optimizer.learning_rate
        ))

        if new_lr < old_lr:
            print(f'\nLR reduced {old_lr:.2e} → {new_lr:.2e} | Reloading BEST weights...\n')
            best_model = load_model(self.checkpoint_path, compile=False, safe_mode=False)
            self.model.set_weights(best_model.get_weights())
            del best_model

---

## 8. Training

**Callbacks:**
- `ModelCheckpoint` — saves best model by `val_loss` to `/kaggle/working/best_model.keras`
- `ReduceLRReloadBest` — factor=0.5, patience=4, min_lr=1e-5 (reloads best weights on reduction)
- `EarlyStopping` — patience=8, restores best weights at end

**Training result (logged from Kaggle run):**
- 23 epochs total
- Best val_loss: **4.33** at epoch 15
- Learning rate was reduced twice during training
- Training on P100 GPU, batch_size=64

In [ ]:
CHECKPOINT_PATH = '/kaggle/working/best_model.keras'

checkpoint_callback = ModelCheckpoint(
    CHECKPOINT_PATH,
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

reduce_lr_callback = ReduceLRReloadBest(
    checkpoint_path=CHECKPOINT_PATH,
    monitor='val_loss',
    factor=0.5,        # halve LR on plateau
    patience=4,        # wait 4 epochs before reducing
    min_lr=1e-5,
    verbose=1
)

early_stopping_callback = EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

In [ ]:
# ── Training ran on Kaggle (P100 GPU). Commented out for inference-only use.
# ── 23 epochs, best val_loss 4.33 at epoch 15.

# history = model.fit(
#     train_dataset,
#     epochs=50,
#     steps_per_epoch=train_steps,
#     validation_data=val_dataset,
#     validation_steps=val_steps,
#     verbose=1,
#     callbacks=[checkpoint_callback, reduce_lr_callback, early_stopping_callback]
# )

# Load best checkpoint for inference
# model = load_model(CHECKPOINT_PATH, compile=False, safe_mode=False)

print('Training section — run on Kaggle.')
print('See Kaggle notebook: https://www.kaggle.com/huzefamerchant/image-captioning-attention')
print('Best result: val_loss=4.33 at epoch 15 (of 23 total)')

---

## 9. Inference — Beam Search

v2 uses **beam search** instead of greedy decoding from v1.

### Why beam search is better than greedy

Greedy decoding makes locally optimal choices: at each step, it picks the single highest
probability token. But a locally optimal choice can lead to a globally poor sequence —
once an early word is wrong, every subsequent word conditions on that mistake.

Beam search maintains `beam_width` candidate sequences simultaneously. At each step,
every beam is extended with the top-k tokens, giving `beam_width × k` candidates.
These are sorted by cumulative log-probability and only the top `beam_width` are kept.

### Length normalization

Without normalization, shorter captions would win because each word multiplication
of probability < 1 shrinks the score. Log-probability sum grows more negative for
longer sequences. Length normalization divides the score by `len(seq) ** alpha`
where `alpha=0.7` is a standard value that balances length preference.

### UNK skipping

Token index 1 (`[UNK]`) is skipped during beam expansion — if UNK is in the top-k,
the search takes the next non-UNK token instead.

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_dict = dict(enumerate(vocab))  # index → word

START_IDX = vocab.index('<start>') if '<start>' in vocab else 2
END_IDX   = vocab.index('<end>')   if '<end>'   in vocab else 3
UNK_IDX   = 1


def caption_generator_beam(image_feature, model, vocab_dict,
                            beam_width=5, max_len=max_len, alpha=0.7):
    """Generate a caption using beam search.

    Args:
        image_feature: np.array (64, 2048) — pre-extracted spatial features
        model: trained Keras captioning model
        vocab_dict: dict {int: str} — index to word mapping
        beam_width: number of beams to maintain
        max_len: maximum caption length
        alpha: length normalization exponent (0.7 standard)

    Returns:
        str: generated caption without <start>/<end> tokens
    """
    image_feature_batch = np.expand_dims(image_feature, axis=0)  # (1, 64, 2048)

    beams = [([START_IDX], 0.0)]  # list of (sequence, cumulative_log_prob)

    for step in range(max_len):
        all_candidates = []

        for seq, score in beams:
            if seq[-1] == END_IDX:
                all_candidates.append((seq, score))
                continue

            seq_input = pad_sequences(
                [seq], maxlen=max_len, padding='post', truncating='post', value=0
            )
            # Model output: (1, max_len, vocab_size) — take prediction at current position
            pred_probs = model.predict([image_feature_batch, seq_input], verbose=0)[0, len(seq) - 1]

            top_indices = np.argsort(pred_probs)[-(beam_width + 1):]  # extra slot for UNK
            top_indices = [idx for idx in top_indices if idx != UNK_IDX][:beam_width]
            top_indices = np.int32(top_indices)

            for idx in top_indices:
                new_seq   = seq + [int(idx)]
                new_score = score + np.log(pred_probs[idx] + 1e-9)
                all_candidates.append((new_seq, new_score))

        # Sort by length-normalized score (higher is better)
        ordered = sorted(
            all_candidates,
            key=lambda x: x[1] / (len(x[0]) ** alpha),
            reverse=True
        )
        beams = ordered[:beam_width]

    best_seq = beams[0][0]
    words = [vocab_dict[i] for i in best_seq[1:] if i != END_IDX]
    return ' '.join(words)

---

## 10. BLEU Evaluation

Same BLEU-4 setup as v1 (SmoothingFunction().method4, single reference per image).
Beam search (width=5) is used for evaluation — it was the strongest decoder available in v2.

In [ ]:
def evaluate_bleu_beam(model, val_keys, image_features_dict_reshaped,
                       filtered_captions_dict, vocab_dict, num_samples=100):
    """Compute average BLEU-4 on a random sample of validation images using beam search."""
    smoothie = SmoothingFunction().method4
    bleu_scores = []

    sample_keys = random.sample(val_keys, min(num_samples, len(val_keys)))

    for path in tqdm(sample_keys):
        image_feature = image_features_dict_reshaped[path]

        pred_text = caption_generator_beam(image_feature, model, vocab_dict)

        ref_text = filtered_captions_dict[path]
        ref_text = re.sub(r'<start>|<end>', '', ref_text).strip().lower()
        reference = [ref_text.split()]

        hypothesis = pred_text.lower().split()

        score = sentence_bleu(
            reference, hypothesis,
            weights=(0.25, 0.25, 0.25, 0.25),
            smoothing_function=smoothie
        )
        bleu_scores.append(score)

    avg = sum(bleu_scores) / len(bleu_scores)
    print(f'Average BLEU-4 ({num_samples} samples, beam_width=5): {avg:.4f}')
    return avg


# ── Run with trained model loaded:
# bleu_v2 = evaluate_bleu_beam(model, val_keys, image_features_dict_reshaped,
#                               filtered_captions_dict, vocab_dict, num_samples=100)
print('BLEU evaluation: run with trained model loaded.')
print('Reported result from Kaggle run: BLEU-4 = 0.026 (beam_width=5)')

---

## 11. Results and Analysis

### Training result
- 23 epochs, best val_loss = **4.33** at epoch 15
- LR reduced twice during training (ReduceLRReloadBest triggered)
- Early stopping at epoch 23 (no improvement for 8 epochs)

### Inference behavior (beam search, width=5)

Sample outputs:
```
Reference:  <start> vday 🖤 ♥ <end>
Generated:  I love you
BLEU-4:     0

Reference:  <start> floors us with yet another divine look for . <end>
Generated:  Thank you for for for for for the beautiful
BLEU-4:     0.021

Reference:  <start> Night with Brogs 💜 <end>
Generated:  I love you
BLEU-4:     0
```

Average BLEU-4: **0.026**

### Post-mortem — 9 problems identified (ChatGPT analysis)

A systematic analysis was done of why v2 still fails despite better architecture:

1. **Captions not visually grounded** — Instagram captions describe feelings, not images.
   Attention over image patches cannot help when the caption is unrelated to visual content.
2. **Single reference per image** — BLEU with 1 reference is extremely harsh. Any synonym
   or paraphrase scores 0. The true model quality may be higher than 0.026 suggests.
3. **Bad-phrase filtering insufficient** — Removing 'i love you' still leaves thousands
   of emotionally-grounded but visually-meaningless captions.
4. **Vocabulary too large for dataset size** — 8k tokens on 20k images means many tokens
   appear < 5 times; the model cannot learn them reliably.
5. **Repetition in beam search** — 'Thank you for for for for' shows the model entering
   a local loop. Repetition penalty (used in v3) was not implemented here.
6. **Add() fusion may be mismatched** — If context and lstm_output are on different scales,
   Add() can cause one branch to dominate the other.
7. **No label smoothing** — Label smoothing (used in v3) prevents overconfident predictions
   that lock the model into specific token choices early in decoding.
8. **Teacher forcing exposure bias** — During training the model always sees ground truth
   previous tokens. At inference, it sees its own (often wrong) previous tokens.
   The distribution mismatch causes error propagation.
9. **Data volume** — 20k images is small for learning vision-language alignment from scratch.
   COCO (118k images, 5 captions each = 590k pairs) has 30× more signal.

**Conclusion:** Architecture improvements (attention, LSTM init, gating) are real progress.
But the data problem is insurmountable on Instagram captions. The fundamental lesson:
image captioning requires visually-grounded captions. The switch to Flickr30k and COCO
in v3 is the fix — and the BLEU jump from 0.026 → 0.134 confirms it.

---

## 12. Key Insights from v2

1. **Spatial features unlock attention, but attention alone cannot fix data quality.**
   Switching `layers[-2]` → `mixed10` is the right architectural move. The attention
   layer does learn to weight different patches differently. But if the training captions
   say 'link in bio' for a sunset photo, no attention map can recover the right words.

2. **LSTM initialization with image context is a meaningful improvement.**
   Giving the LSTM a visual starting point (h0, c0 from GlobalAveragePooling)
   means every word is generated from a state that has already seen the image.
   This pattern appears in several published attention models (e.g., Show, Attend and Tell).

3. **Custom callbacks can combine LR scheduling with state management.**
   `ReduceLRReloadBest` is a non-trivial callback that fixes a real training instability.
   The pattern — compare LR before/after parent call — is reusable for any ReduceLROnPlateau
   scenario where model state matters at LR reduction time.

4. **Masked loss is necessary, not optional.**
   Without masking, the loss function is partially optimizing to predict zero (padding).
   This dilutes gradients and slows learning. Every sequence-to-sequence model needs this.

5. **Framework (TF/Keras) mattered less than data. PyTorch vs TF was not the bottleneck.**
   The switch to PyTorch in v3 was driven by ecosystem preference (timm, transformers libraries)
   and the desire to use pretrained GPT-2. The BLEU jump in v3 comes from data + architecture,
   not from the framework change.

---

## 13. Version Comparison Table

| Version | Encoder | Decoder | Dataset | BLEU-4 | Key Problem |
|---------|---------|---------|---------|--------|-------------|
| **v0 (Colab)** | InceptionV3 `layers[-2]` flat 2048 | LSTM(256) | Instagram | — | Runtime crash: `train_dataset not defined` |
| v1 | InceptionV3 `layers[-2]` flat 2048 | LSTM(512), Add/Concat | Instagram | ~0.0 | Captions not visually grounded |
| **v2 (this notebook)** | InceptionV3 `mixed10` spatial (64×2048) | TemporalImageAttention + LSTM(256) + image init + gating | Instagram | **0.026** | Same data problem; attention helps but cannot fix it |
| v3 Phase 1 | ConvNeXt-Tiny multi-scale (64,768) | Custom Transformer Decoder | Flickr30k | 0.0674 | Framework switch to PyTorch; better data |
| v3 Phase 2d | ConvNeXt + Perceiver | GPT-2 + CrossAttention (×12) | Flickr30k | 0.0656 | Cross-attention recovers from prepend regression |
| **v3 Phase 3a** | ConvNeXt + Perceiver | GPT-2 + CrossAttention (×12) | **COCO 2017** | **0.1341** | **Best model — COCO's visual captions unlock the jump** |

---

**Next:** `v3a_convnext_transformer.ipynb` — complete framework switch to PyTorch,
ConvNeXt-Tiny multi-scale encoder, and a custom Transformer decoder from scratch.